# 10 FS2 Benchmark And Shortlisting

This is the combined `FS2` comparison stage.

Run this notebook after `04`, `05`, `06`, and `09` have produced fresh benchmark artifacts. It builds the shared `model_comparison` run used by the final conclusion notebook.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


In [ ]:
rows = []
for run_label in ['naive_benchmark', 'lear_benchmark', 'xgboost_benchmark', 'prophet_benchmark', 'model_comparison']:
    run_dir = latest_run_or_none(run_label)
    rows.append(
        {
            "run_label": run_label,
            "latest_run": str(run_dir) if run_dir is not None else "not yet run",
        }
    )

display(pd.DataFrame(rows))


In [ ]:
display(shortlisting_policy_frame())


In [ ]:
run_dir = latest_run_or_none("model_comparison")

if run_dir is None:
    print("No combined model comparison run exists yet under the finalized methodology.")
else:
    print(run_dir)
    metrics_by_reporting_level = load_csv(run_dir, "metrics_by_reporting_level.csv")
    official_naive = load_json(run_dir, "official_naive_reference.json")
    focus_models = [
        "naive_previous_week",
        "naive_previous_year",
        "lear_fs1",
        "xgboost_fs1",
        "lear_fs2",
        "xgboost_fs2",
        "prophet_fs2",
    ]
    model_order = {model_name: position for position, model_name in enumerate(focus_models)}
    display(pd.DataFrame([official_naive]))
    display(
        metrics_by_reporting_level[metrics_by_reporting_level["model"].isin(focus_models)]
        .assign(_model_order=lambda frame: frame["model"].map(model_order).fillna(len(model_order)))
        .sort_values(["dataset_split", "reporting_level_sort_order", "_model_order", "model"])
        .drop(columns=["_model_order"])
        .reset_index(drop=True)
    )


## Expanded comparison reporting

The lightweight run tables above are kept as-is. The sections below restore the richer notebook reporting layer that compares the saved run against the relevant benchmark and peer models, then reuses the frozen objective weeks.


In [ ]:
from IPython.display import Image

from hourly_da.notebook_support import (
    apply_standard_matplotlib_style,
    build_compared_models_overview,
    build_dm_summary_table,
    build_model_style_map,
    build_reporting_summary_table,
    build_runtime_summary_table,
    build_week_metrics_for_predictions,
    filter_available_comparison_specs,
    load_standard_report_bundle,
    render_plot_gallery,
    style_dm_summary_table,
    style_model_overview_table,
    style_reporting_summary_table,
    style_runtime_summary_table,
    write_actual_vs_predicted_scatter_plot,
    write_horizon_error_plot,
    write_mae_by_hour_of_day_plot,
    write_residual_distribution_plot,
    write_standard_week_selection_plots,
)

run_dir = latest_run_or_none("model_comparison")
report_bundle = None
model_styles = {}
report_output_dir = None
official_naive_model = None
HORIZON_PLOT_MODELS = []
WEEK_PLOT_MODELS = []
DIAGNOSTIC_MODELS = []
DM_CHALLENGER_MODELS = []

if run_dir is None:
    print("No saved run exists yet for this notebook under the finalized methodology.")
else:
    apply_standard_matplotlib_style()
    current_suite_models = load_json(run_dir, "suite_models.json").get("models", [])
    current_model_names = [
        record.get("model") or record.get("name")
        for record in current_suite_models
        if (record.get("model") or record.get("name"))
    ]
    active_naive_models = set(naive_model_names())
    current_model_names = [
        model_name
        for model_name in current_model_names
        if not (str(model_name).startswith("naive_") and model_name not in active_naive_models)
    ]
    official_naive = load_json(run_dir, "official_naive_reference.json")
    official_naive_model = str(official_naive["model"])
    official_naive_display = f"Naive benchmark ({official_naive_model.replace('naive_', '').replace('_', ' ').title()})"
    comparison_specs = [
        {
            "model": official_naive_model,
            "display_name": official_naive_display,
            "description": "Validation-selected FS0 benchmark for the combined FS2 comparison.",
            "role": "benchmark",
        },
    ]
    comparison_specs.extend(
        filter_available_comparison_specs(
            output_root=output_root,
            current_run_dir=run_dir,
            candidate_specs=[
                {
                    "model": "lear_fs1",
                    "source": "current",
                    "display_name": "LEAR FS1",
                    "description": "Earlier FS1 linear benchmark from notebook 05.",
                    "role": "prior",
                },
                {
                    "model": "xgboost_fs1",
                    "source": "current",
                    "display_name": "XGBoost FS1",
                    "description": "Earlier FS1 tree benchmark from notebook 06.",
                    "role": "prior",
                },
                {
                    "model": "lear_fs2",
                    "source": "current",
                    "display_name": "LEAR FS2",
                    "description": "FS2 linear benchmark from notebook 07.",
                    "role": "current",
                },
                {
                    "model": "xgboost_fs2",
                    "source": "current",
                    "display_name": "XGBoost FS2",
                    "description": "FS2 tree benchmark from notebook 08.",
                    "role": "current",
                },
                {
                    "model": "prophet_fs2",
                    "source": "current",
                    "display_name": "Prophet FS2",
                    "description": "FS2 additive benchmark from notebook 09.",
                    "role": "current",
                },
            ],
        )
    )
    seen_models = set()
    COMPARISON_MODEL_SPECS = []
    for spec in comparison_specs:
        model_name = str(spec["model"])
        if model_name in seen_models:
            continue
        seen_models.add(model_name)
        COMPARISON_MODEL_SPECS.append(spec)

    HORIZON_PLOT_MODELS = [spec["model"] for spec in COMPARISON_MODEL_SPECS]
    WEEK_PLOT_MODELS = HORIZON_PLOT_MODELS.copy()
    DIAGNOSTIC_MODELS = WEEK_PLOT_MODELS.copy()
    DM_CHALLENGER_MODELS = [model_name for model_name in HORIZON_PLOT_MODELS if model_name != official_naive_model]
    if not COMPARISON_MODEL_SPECS:
        print("No comparison models were available for the expanded reporting block.")
    else:
        report_bundle = load_standard_report_bundle(
            output_root=output_root,
            current_run_dir=run_dir,
            comparison_specs=COMPARISON_MODEL_SPECS,
        )
        model_styles = build_model_style_map(
            report_bundle["comparison_specs"],
            official_naive_model=str(report_bundle["official_naive"]["model"]),
        )
        report_output_dir = output_root / "notebook_artifacts" / "10_fs2_benchmark_and_shortlisting" / run_dir.name / "standard_report"
        report_output_dir.mkdir(parents=True, exist_ok=True)

        print(f"Current run: {run_dir.name}")
        print(f"Current models: {current_model_names}")
        print(f"Naive benchmark for later comparisons: {official_naive_model}")
        print(f"Expanded comparison models: {[spec['model'] for spec in COMPARISON_MODEL_SPECS]}")


## 1. Short overview of compared models


In [ ]:
if report_bundle is None:
    print("Expanded comparison reporting is not available yet for this notebook.")
else:
    overview_table = build_compared_models_overview(
        report_bundle["comparison_specs"],
        official_naive_model=str(report_bundle["official_naive"]["model"]),
    )
    display(style_model_overview_table(overview_table))
    display(pd.DataFrame([report_bundle["official_naive"]]))


## 2. Main validation and test summary tables

Each table reports the same thesis metrics for the three standard reporting slices:
- `D only`
- `Full-horizon`


In [ ]:
if report_bundle is None:
    print("Expanded summary tables are not available yet for this notebook.")
else:
    validation_summary = build_reporting_summary_table(
        report_bundle["metrics_by_reporting_level"],
        split_name="validation",
        model_order=report_bundle["model_order"],
    )
    test_summary = build_reporting_summary_table(
        report_bundle["metrics_by_reporting_level"],
        split_name="test",
        model_order=report_bundle["model_order"],
    )

    if not validation_summary.empty:
        display(style_reporting_summary_table(validation_summary, caption="Validation summary"))
    if not test_summary.empty:
        display(style_reporting_summary_table(test_summary, caption="Test summary"))


## 3. By-horizon error view

This figure keeps the split fixed and shows how MAE changes from `D` through `D+4`.


In [ ]:
if report_bundle is None or report_output_dir is None:
    print("Expanded horizon plots are not available yet for this notebook.")
else:
    horizon_plot_path = write_horizon_error_plot(
        metrics_by_lead_day=report_bundle["metrics_by_lead_day"][
            report_bundle["metrics_by_lead_day"]["model"].isin(HORIZON_PLOT_MODELS)
        ].copy(),
        output_path=report_output_dir / "horizon_error_mae.png",
        model_order=HORIZON_PLOT_MODELS,
        model_styles=model_styles,
    )
    if horizon_plot_path is not None:
        display(Image(filename=str(horizon_plot_path)))


## 4. Forecast vs actual on the frozen week selections

These plots reuse the objectively selected weeks from notebook `02`. For a clean visual comparison, the overlays use the operational `D only` path so each target hour appears once.


In [ ]:
if report_bundle is None or report_output_dir is None:
    print("Expanded selected-week reporting is not available yet for this notebook.")
else:
    try:
        selection_run_dir, selected_weeks = load_selected_case_weeks(output_root)
    except FileNotFoundError:
        print("No saved objective week selection artifact exists yet.")
    else:
        print(selection_run_dir)
        display(selected_weeks[["category", "iso_week_id", "week_start_local_date", "week_end_local_date"]])

        model_label_map = (
            report_bundle["comparison_specs"][["model", "display_name"]]
            .drop_duplicates(subset=["model"])
            .set_index("model")["display_name"]
            .to_dict()
        )
        week_metrics = build_week_metrics_for_predictions(report_bundle["predictions_long"], config, selected_weeks)
        week_model_order = {model_name: position for position, model_name in enumerate(WEEK_PLOT_MODELS)}
        week_metrics_display = (
            week_metrics[week_metrics["model"].isin(WEEK_PLOT_MODELS)]
            .assign(
                Model=lambda frame: frame["model"].map(model_label_map).fillna(frame["model"]),
                _model_order=lambda frame: frame["model"].map(week_model_order).fillna(len(week_model_order)),
            )
            [["category", "iso_week_id", "_model_order", "Model", "mae", "rmse", "bias", "coverage_pct", "max_abs_error"]]
            .sort_values(["category", "_model_order", "Model"])
            .drop(columns=["_model_order"])
            .reset_index(drop=True)
        )
        display(
            week_metrics_display.style
            .format(
                {
                    "mae": "{:.2f}",
                    "rmse": "{:.2f}",
                    "bias": "{:+.2f}",
                    "coverage_pct": "{:.2f}%",
                    "max_abs_error": "{:.2f}",
                }
            )
            .hide(axis="index")
        )

        week_plot_paths = write_standard_week_selection_plots(
            predictions=report_bundle["predictions_long"],
            config=config,
            selected_weeks=selected_weeks,
            output_dir=report_output_dir / "week_plots",
            model_order=WEEK_PLOT_MODELS,
            model_styles=model_styles,
            split_name="test",
            reporting_level="d_only",
            title_prefix='FS2 benchmark and shortlisting comparison, D-only path',
        )
        display(render_plot_gallery(week_plot_paths, columns=2))


## 5. Diagnostic plots

The diagnostics below focus on the **test split** and the same `D only` operational path as the week overlays, so the interpretation is based on unique forecast-target pairs rather than repeated multi-origin horizons.


In [ ]:
if report_bundle is None or report_output_dir is None:
    print("Expanded diagnostic plots are not available yet for this notebook.")
else:
    diagnostic_plot_paths = []

    mae_by_hour_path = write_mae_by_hour_of_day_plot(
        predictions=report_bundle["predictions_long"],
        config=config,
        output_path=report_output_dir / "diagnostics" / "mae_by_hour_of_day.png",
        model_order=DIAGNOSTIC_MODELS,
        model_styles=model_styles,
    )
    if mae_by_hour_path is not None:
        diagnostic_plot_paths.append(mae_by_hour_path)

    residual_path = write_residual_distribution_plot(
        predictions=report_bundle["predictions_long"],
        config=config,
        output_path=report_output_dir / "diagnostics" / "residual_distribution.png",
        model_order=DIAGNOSTIC_MODELS,
        model_styles=model_styles,
    )
    if residual_path is not None:
        diagnostic_plot_paths.append(residual_path)

    scatter_path = write_actual_vs_predicted_scatter_plot(
        predictions=report_bundle["predictions_long"],
        config=config,
        output_path=report_output_dir / "diagnostics" / "actual_vs_predicted.png",
        model_order=DIAGNOSTIC_MODELS,
        model_styles=model_styles,
    )
    if scatter_path is not None:
        diagnostic_plot_paths.append(scatter_path)

    display(render_plot_gallery(diagnostic_plot_paths, columns=2))


## 6. Statistical comparison

The Diebold-Mariano table is interpreted as follows:
- negative DM statistic favors the challenger
- positive DM statistic favors the benchmark
- the verdict column applies a `p < 0.05` threshold


In [ ]:
if report_bundle is None:
    print("Expanded Diebold-Mariano reporting is not available yet for this notebook.")
else:
    dm_display_order = [
        row["display_name"]
        for row in report_bundle["comparison_specs"].to_dict(orient="records")
        if row["model"] in DM_CHALLENGER_MODELS
    ]
    dm_summary = build_dm_summary_table(
        report_bundle["diebold_mariano_by_reporting_level"][
            report_bundle["diebold_mariano_by_reporting_level"]["challenger_model"].isin(DM_CHALLENGER_MODELS)
        ].copy(),
        challenger_display_order=dm_display_order,
    )
    if dm_summary.empty:
        print("No configured Diebold-Mariano comparisons were available for this notebook.")
    else:
        display(style_dm_summary_table(dm_summary))


## 7. Runtime and practicality summary


In [ ]:
if report_bundle is None:
    print("Expanded runtime reporting is not available yet for this notebook.")
else:
    runtime_summary = build_runtime_summary_table(
        report_bundle["timing_summary"],
        model_order=report_bundle["model_order"],
    )
    if runtime_summary.empty:
        print("No runtime summary was available for this notebook.")
    else:
        display(style_runtime_summary_table(runtime_summary))


## Optional execution hook


In [ ]:
ALLOW_HEAVY_RERUN = False

if ALLOW_HEAVY_RERUN:
    estimate = estimate_run_duration_seconds(output_root, "model_comparison")
    if estimate is not None:
        print(
            "Heavy rerun warning: latest comparable run "
            f"{estimate['run_id']} suggests about {format_duration(float(estimate['estimate_seconds']))}."
        )
    else:
        print("Heavy rerun warning: no comparable runtime estimate was found for this stage.")

    command = [sys.executable, str(PACKAGE_ROOT / "run_model_comparison.py")]

    started = time.perf_counter()
    subprocess.run(command, check=True)
    elapsed_seconds = time.perf_counter() - started
    print(f"Actual wall-clock time: {format_duration(elapsed_seconds)}")
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True only when you are ready to execute the finalized pipeline.")
